In [4]:
!pip install openai pandas numpy langchain langchain-openai langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [9]:
import os
import json
import pandas as pd
import numpy as np
from openai import OpenAI

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"

# ── Simple Memory (no LangChain needed) ──────────────────────
# Stores past investigations as a list of dicts
# Before each new run, we compress them into a short summary
# using a separate GPT call — same idea as ConversationSummaryMemory

investigation_memory = []   # grows after each run_agent() call

def save_to_memory(goal: str, conclusion: str):
    investigation_memory.append({"goal": goal, "conclusion": conclusion})
    print(f"💾 Saved to memory. Total investigations: {len(investigation_memory)}")

def get_memory_context() -> str:
    if not investigation_memory:
        return ""
    # Ask GPT to compress all past investigations into 3-4 sentences
    history_text = "\n\n".join([
        f"Investigation {i+1}:\nGoal: {m['goal']}\nConclusion: {m['conclusion']}"
        for i, m in enumerate(investigation_memory)
    ])
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Summarise these past investigations in 3-4 sentences. Focus on what problems were found, when they started, and which segments were affected."},
            {"role": "user",   "content": history_text}
        ],
        temperature=0
    )
    return response.choices[0].message.content

print("✅ Setup complete")
print("✅ Custom memory initialised")

✅ Setup complete
✅ Custom memory initialised


In [10]:
def build_funnel_dataset():
    np.random.seed(42)
    users = 10000

    traffic_source = np.random.choice(["ads", "organic"], users, p=[0.4, 0.6])
    device         = np.random.choice(["mobile", "desktop"], users, p=[0.7, 0.3])
    experiment     = np.random.choice(["A", "B"], users)
    date           = pd.to_datetime("2025-01-01") + pd.to_timedelta(
        np.random.randint(0, 30, users), unit="D"
    )

    stage = []
    for _ in range(users):
        if np.random.rand() > 0.70:
            stage.append("visit")
            continue
        if np.random.rand() > 0.50:
            stage.append("signup")
            continue
        if np.random.rand() > 0.40:
            stage.append("activation")
            continue
        stage.append("purchase")

    df = pd.DataFrame({
        "user_id"        : np.arange(users),
        "date"           : date,
        "traffic_source" : traffic_source,
        "device"         : device,
        "experiment"     : experiment,
        "stage_reached"  : stage
    })

    # Hidden anomaly: mobile checkout broke after Jan 20
    mask = (
        (df["device"] == "mobile") &
        (df["stage_reached"] == "purchase") &
        (df["date"] > "2025-01-20")
    )
    df.loc[mask, "stage_reached"] = "activation"

    stage_order = {"visit": 1, "signup": 2, "activation": 3, "purchase": 4}
    df["stage_number"] = df["stage_reached"].map(stage_order)

    return df

df = build_funnel_dataset()
print("✅ Dataset built:", df.shape)
print(df["stage_reached"].value_counts())

✅ Dataset built: (10000, 7)
stage_reached
signup        3465
visit         2986
activation    2470
purchase      1079
Name: count, dtype: int64


In [11]:
def get_funnel_summary() -> str:
    total       = len(df)
    visits      = total
    signups     = int((df["stage_number"] >= 2).sum())
    activations = int((df["stage_number"] >= 3).sum())
    purchases   = int((df["stage_number"] >= 4).sum())

    summary = {
        "total_users"                 : visits,
        "signups"                     : signups,
        "activations"                 : activations,
        "purchases"                   : purchases,
        "visit_to_signup_rate"        : round(signups / visits, 3),
        "signup_to_activation_rate"   : round(activations / signups, 3),
        "activation_to_purchase_rate" : round(purchases / activations, 3),
        "overall_conversion_rate"     : round(purchases / visits, 3)
    }
    return json.dumps(summary)


def detect_anomalies() -> str:
    daily = df.groupby("date").agg(
        activations=("stage_number", lambda x: (x >= 3).sum()),
        purchases  =("stage_number", lambda x: (x >= 4).sum())
    ).reset_index()

    daily["rate"] = daily["purchases"] / daily["activations"].replace(0, np.nan)
    avg_rate  = daily["rate"].mean()
    threshold = avg_rate * 0.70
    anomalies = daily[daily["rate"] < threshold].copy()
    anomalies["date"] = anomalies["date"].astype(str)

    result = {
        "average_activation_purchase_rate" : round(avg_rate, 3),
        "anomaly_threshold"                : round(threshold, 3),
        "anomaly_days_count"               : len(anomalies),
        "anomaly_dates"                    : anomalies["date"].tolist(),
        "anomaly_rates"                    : anomalies["rate"].round(3).tolist()
    }
    return json.dumps(result)


def get_device_breakdown(start_date: str = None, end_date: str = None) -> str:
    filtered = df.copy()
    if start_date:
        filtered = filtered[filtered["date"] >= start_date]
    if end_date:
        filtered = filtered[filtered["date"] <= end_date]

    breakdown = {}
    for device in ["mobile", "desktop"]:
        d           = filtered[filtered["device"] == device]
        activations = int((d["stage_number"] >= 3).sum())
        purchases   = int((d["stage_number"] >= 4).sum())
        rate        = round(purchases / activations, 3) if activations > 0 else 0
        breakdown[device] = {
            "activations"              : activations,
            "purchases"                : purchases,
            "activation_purchase_rate" : rate
        }
    return json.dumps(breakdown)


def get_traffic_source_breakdown(start_date: str = None, end_date: str = None) -> str:
    filtered = df.copy()
    if start_date:
        filtered = filtered[filtered["date"] >= start_date]
    if end_date:
        filtered = filtered[filtered["date"] <= end_date]

    breakdown = {}
    for source in ["ads", "organic"]:
        d           = filtered[filtered["traffic_source"] == source]
        activations = int((d["stage_number"] >= 3).sum())
        purchases   = int((d["stage_number"] >= 4).sum())
        rate        = round(purchases / activations, 3) if activations > 0 else 0
        breakdown[source] = {
            "activations"              : activations,
            "purchases"                : purchases,
            "activation_purchase_rate" : rate
        }
    return json.dumps(breakdown)

# Quick test
print("=== TOOL 1: Funnel Summary ===")
print(get_funnel_summary())

print("\n=== TOOL 2: Anomaly Detection ===")
print(detect_anomalies())

print("\n=== TOOL 3: Device Breakdown (after Jan 20) ===")
print(get_device_breakdown(start_date="2025-01-20"))

print("\n=== TOOL 4: Traffic Source Breakdown ===")
print(get_traffic_source_breakdown())

=== TOOL 1: Funnel Summary ===
{"total_users": 10000, "signups": 7014, "activations": 3549, "purchases": 1079, "visit_to_signup_rate": 0.701, "signup_to_activation_rate": 0.506, "activation_to_purchase_rate": 0.304, "overall_conversion_rate": 0.108}

=== TOOL 2: Anomaly Detection ===
{"average_activation_purchase_rate": 0.306, "anomaly_threshold": 0.214, "anomaly_days_count": 10, "anomaly_dates": ["2025-01-21", "2025-01-22", "2025-01-23", "2025-01-24", "2025-01-25", "2025-01-26", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30"], "anomaly_rates": [0.161, 0.079, 0.13, 0.158, 0.125, 0.129, 0.051, 0.159, 0.156, 0.105]}

=== TOOL 3: Device Breakdown (after Jan 20) ===
{"mobile": {"activations": 930, "purchases": 39, "activation_purchase_rate": 0.042}, "desktop": {"activations": 430, "purchases": 171, "activation_purchase_rate": 0.398}}

=== TOOL 4: Traffic Source Breakdown ===
{"ads": {"activations": 1436, "purchases": 447, "activation_purchase_rate": 0.311}, "organic": {"activations

In [12]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name"        : "get_funnel_summary",
            "description" : (
                "Returns overall funnel conversion rates at each stage. "
                "Call this FIRST to understand baseline funnel performance. "
                "Do NOT draw conclusions from this tool alone — always follow up "
                "with anomaly detection before making any diagnosis."
            ),
            "parameters"  : {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name"        : "detect_anomalies",
            "description" : (
                "Detects dates where activation to purchase rate dropped more than "
                "30% below average. Returns anomaly dates and rates. "
                "Call this SECOND to find WHEN the problem started. "
                "If no anomalies are found, do NOT assume there is a problem — "
                "report that metrics look normal."
            ),
            "parameters"  : {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name"        : "get_device_breakdown",
            "description" : (
                "Returns conversion rates split by mobile vs desktop. "
                "Use start_date from anomaly detection to filter to the affected period. "
                "If no anomaly date was found, call without date filters."
            ),
            "parameters"  : {
                "type"       : "object",
                "properties" : {
                    "start_date" : {"type": "string", "description": "YYYY-MM-DD. Optional."},
                    "end_date"   : {"type": "string", "description": "YYYY-MM-DD. Optional."}
                },
                "required" : []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name"        : "get_traffic_source_breakdown",
            "description" : (
                "Returns conversion rates split by ads vs organic traffic. "
                "Call this to rule out marketing or acquisition causes. "
                "If both channels are equally affected, the problem is product "
                "or technical, not a marketing issue."
            ),
            "parameters"  : {
                "type"       : "object",
                "properties" : {
                    "start_date" : {"type": "string", "description": "YYYY-MM-DD. Optional."},
                    "end_date"   : {"type": "string", "description": "YYYY-MM-DD. Optional."}
                },
                "required" : []
            }
        }
    }
]

print(f"✅ Tool registry loaded: {len(TOOLS)} tools registered")
for tool in TOOLS:
    print(f"   - {tool['function']['name']}: {tool['function']['description'][:60]}...")

✅ Tool registry loaded: 4 tools registered
   - get_funnel_summary: Returns overall funnel conversion rates at each stage. Call ...
   - detect_anomalies: Detects dates where activation to purchase rate dropped more...
   - get_device_breakdown: Returns conversion rates split by mobile vs desktop. Use sta...
   - get_traffic_source_breakdown: Returns conversion rates split by ads vs organic traffic. Ca...


In [13]:
def dispatch_tool(tool_name: str, tool_args: dict) -> str:
    print(f"\n  ⚙️  TOOL CALLED  : {tool_name}")
    print(f"  📥 ARGUMENTS    : {tool_args}")

    if tool_name == "get_funnel_summary":
        result = get_funnel_summary()
    elif tool_name == "detect_anomalies":
        result = detect_anomalies()
    elif tool_name == "get_device_breakdown":
        result = get_device_breakdown(**tool_args)
    elif tool_name == "get_traffic_source_breakdown":
        result = get_traffic_source_breakdown(**tool_args)
    else:
        result = json.dumps({
            "error"           : f"Tool '{tool_name}' does not exist.",
            "available_tools" : [
                "get_funnel_summary",
                "detect_anomalies",
                "get_device_breakdown",
                "get_traffic_source_breakdown"
            ]
        })

    print(f"  📤 RESULT PREVIEW: {result[:120]}...")
    return result

# Quick test
print("=== DISPATCHER TEST 1: Valid tool ===")
dispatch_tool("get_funnel_summary", {})

print("\n=== DISPATCHER TEST 2: Tool with arguments ===")
dispatch_tool("get_device_breakdown", {"start_date": "2025-01-20"})

print("\n=== DISPATCHER TEST 3: Hallucinated tool (safety net) ===")
dispatch_tool("get_geo_breakdown", {})

=== DISPATCHER TEST 1: Valid tool ===

  ⚙️  TOOL CALLED  : get_funnel_summary
  📥 ARGUMENTS    : {}
  📤 RESULT PREVIEW: {"total_users": 10000, "signups": 7014, "activations": 3549, "purchases": 1079, "visit_to_signup_rate": 0.701, "signup_t...

=== DISPATCHER TEST 2: Tool with arguments ===

  ⚙️  TOOL CALLED  : get_device_breakdown
  📥 ARGUMENTS    : {'start_date': '2025-01-20'}
  📤 RESULT PREVIEW: {"mobile": {"activations": 930, "purchases": 39, "activation_purchase_rate": 0.042}, "desktop": {"activations": 430, "pu...

=== DISPATCHER TEST 3: Hallucinated tool (safety net) ===

  ⚙️  TOOL CALLED  : get_geo_breakdown
  📥 ARGUMENTS    : {}
  📤 RESULT PREVIEW: {"error": "Tool 'get_geo_breakdown' does not exist.", "available_tools": ["get_funnel_summary", "detect_anomalies", "get...


'{"error": "Tool \'get_geo_breakdown\' does not exist.", "available_tools": ["get_funnel_summary", "detect_anomalies", "get_device_breakdown", "get_traffic_source_breakdown"]}'

In [14]:
# Confidence levels that trigger human review
HUMAN_REVIEW_THRESHOLD = {"LOW"}

def parse_confidence(final_answer: str) -> str:
    """
    Reads the agent's final answer and extracts the confidence level.
    The agent will be instructed to always end with CONFIDENCE: HIGH/MEDIUM/LOW
    """
    for line in final_answer.upper().split("\n"):
        if "CONFIDENCE:" in line:
            if "HIGH"   in line: return "HIGH"
            if "MEDIUM" in line: return "MEDIUM"
            if "LOW"    in line: return "LOW"
    return "UNKNOWN"


def human_override_gate(confidence: str, final_answer: str) -> bool:
    """
    Pauses execution and asks for human approval if confidence is LOW.

    Returns:
        True  — human approved, proceed
        False — human rejected, do not act on recommendation
    """
    if confidence not in HUMAN_REVIEW_THRESHOLD:
        # HIGH or MEDIUM — no gate needed, proceed automatically
        return True

    print("\n" + "⚠️  " * 15)
    print("⚠️  HUMAN REVIEW REQUIRED")
    print("⚠️  Agent confidence is LOW — recommendation may be unreliable")
    print("⚠️  " * 15)
    print("\nAgent conclusion:")
    print(final_answer)
    print()

    response = input("Proceed with this recommendation? (yes / no): ").strip().lower()

    if response in ["yes", "y"]:
        print("✅ Human approved. Proceeding.")
        return True
    else:
        print("🚫 Human rejected. Recommendation will NOT be acted on.")
        print("   Next steps: investigate manually or re-run with more data.")
        return False


# Quick test — simulate what each confidence level does
print("=== TEST: HIGH confidence (no gate) ===")
result = human_override_gate("HIGH", "Mobile checkout broke on Jan 21.")
print(f"Approved: {result}")

print("\n=== TEST: MEDIUM confidence (no gate) ===")
result = human_override_gate("MEDIUM", "Possible mobile issue but signals mixed.")
print(f"Approved: {result}")

print("\n=== TEST: LOW confidence (gate fires — type yes or no) ===")
result = human_override_gate("LOW", "Cannot determine root cause — no relevant tools.")
print(f"Approved: {result}")

=== TEST: HIGH confidence (no gate) ===
Approved: True

=== TEST: MEDIUM confidence (no gate) ===
Approved: True

=== TEST: LOW confidence (gate fires — type yes or no) ===

⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  
⚠️  HUMAN REVIEW REQUIRED
⚠️  Agent confidence is LOW — recommendation may be unreliable
⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  

Agent conclusion:
Cannot determine root cause — no relevant tools.

Proceed with this recommendation? (yes / no): no
🚫 Human rejected. Recommendation will NOT be acted on.
   Next steps: investigate manually or re-run with more data.
Approved: False


In [15]:
FAILURE_CASES = [

    {
        "id"                : "F1",
        "label"             : "Vague question — no funnel signal",
        "goal"              : (
            "Our revenue dropped last week. "
            "It might be pricing. It might be a competitor. "
            "Investigate and tell me the root cause."
        ),
        "expected_behaviour": (
            "Agent should note that its tools only cover funnel conversion metrics. "
            "It cannot diagnose pricing or competitive issues. "
            "Should flag LOW confidence and recommend manual investigation."
        )
    },

    {
        "id"                : "F2",
        "label"             : "Missing dimension — no geographic tool",
        "goal"              : (
            "Our conversion dropped specifically in Southeast Asia. "
            "Investigate which countries are most affected."
        ),
        "expected_behaviour": (
            "Agent has no geographic breakdown tool. "
            "Should acknowledge this, not hallucinate country-level data. "
            "Should flag LOW confidence and recommend building a geo breakdown tool."
        )
    },

    {
        "id"                : "F3",
        "label"             : "Contradictory signals — no clear root cause",
        "goal"              : (
            "Something seems off with our funnel but I cannot pinpoint what. "
            "Investigate everything and give me a definitive root cause."
        ),
        "expected_behaviour": (
            "Agent should run all tools. "
            "It will find the mobile anomaly — but the goal says 'something seems off' "
            "with no specific complaint. Agent should report what it found "
            "but flag MEDIUM confidence since the question itself was vague."
        )
    }

]

print(f"✅ Failure cases defined: {len(FAILURE_CASES)}")
for case in FAILURE_CASES:
    print(f"\n   {case['id']}: {case['label']}")
    print(f"   Expected: {case['expected_behaviour'][:80]}...")

✅ Failure cases defined: 3

   F1: Vague question — no funnel signal
   Expected: Agent should note that its tools only cover funnel conversion metrics. It cannot...

   F2: Missing dimension — no geographic tool
   Expected: Agent has no geographic breakdown tool. Should acknowledge this, not hallucinate...

   F3: Contradictory signals — no clear root cause
   Expected: Agent should run all tools. It will find the mobile anomaly — but the goal says ...


In [21]:
def build_system_prompt() -> str:
    """
    Builds the system prompt for the agent.
    Pulls in memory context from past investigations.
    """

    memory_context = get_memory_context()

    prompt = f"""You are a senior product analytics agent.

SCOPE — what you are allowed to reason about:
- Funnel conversion rates at each stage
- Anomaly detection on time-series funnel data
- Segment breakdowns by device and traffic source

GUARDRAILS — if asked about ANYTHING outside this scope:
- Pricing, competitors, geography, revenue forecasting,
  marketing strategy, engineering decisions, or any dimension
  you have no tool for — you MUST respond with CONFIDENCE: LOW
  and clearly state: "This question is outside my tool coverage."
- Do NOT speculate or guess. Do NOT use data from one dimension
  to answer a question about a different dimension.

INVESTIGATION PROTOCOL:
1. FIRST check past investigation memory — if the root cause is already
   confirmed in memory, skip funnel summary and anomaly detection.
   Go directly to the specific tool needed to answer the current question.
2. If no relevant memory exists, call get_funnel_summary FIRST
3. Call detect_anomalies SECOND — find when the problem started
4. Use anomaly dates as filters when calling segment breakdowns
5. Call get_device_breakdown and get_traffic_source_breakdown
6. Use ALL relevant tools before writing your final answer
7. Do NOT conclude there is a problem unless anomaly detection confirms one

AVOIDING FALSE CONCLUSIONS:
- Do NOT attribute a problem to a segment unless that segment's
  data clearly differs from the healthy segment
- If signals are contradictory or ambiguous, say so explicitly
  Do not force a conclusion — report the ambiguity

CONFIDENCE SCORING — end EVERY final answer with:

CONFIDENCE: HIGH
REASON: [why — e.g. clear anomaly, strong segment signal]

or CONFIDENCE: MEDIUM
REASON: [what is uncertain]

or CONFIDENCE: LOW
REASON: [why you cannot conclude — missing tools, contradiction, out of scope]

PAST INVESTIGATION MEMORY:
{memory_context if memory_context else "No previous investigations on record."}

Use memory to:
- Avoid re-investigating already confirmed root causes
- Flag if a new pattern differs from past findings
- Note if this looks like a recurring issue
"""
    return prompt

print("✅ System prompt builder defined")
print("\n--- PREVIEW (first 500 chars) ---")
print(build_system_prompt()[:500])

✅ System prompt builder defined

--- PREVIEW (first 500 chars) ---
You are a senior product analytics agent.

SCOPE — what you are allowed to reason about:
- Funnel conversion rates at each stage
- Anomaly detection on time-series funnel data
- Segment breakdowns by device and traffic source

GUARDRAILS — if asked about ANYTHING outside this scope:
- Pricing, competitors, geography, revenue forecasting,
  marketing strategy, engineering decisions, or any dimension
  you have no tool for — you MUST respond with CONFIDENCE: LOW
  and clearly state: "This question


In [17]:
def run_agent(user_goal: str, max_iterations: int = 10) -> str:
    """
    Runs the ReAct agent loop with:
    - Memory context from past investigations
    - Confidence scoring on final answer
    - Human override gate on LOW confidence
    """

    print("\n" + "="*60)
    print("🤖 AGENT STARTED (Stage 2)")
    print(f"🎯 GOAL: {user_goal}")
    print("="*60)

    # --------------------------------------------------------
    # Build messages — system prompt includes memory
    # --------------------------------------------------------
    messages = [
        {"role": "system", "content": build_system_prompt()},
        {"role": "user",   "content": user_goal}
    ]

    reasoning_trace = []
    iteration       = 0

    # ============================================================
    # THE AGENT LOOP
    # ============================================================
    while iteration < max_iterations:
        iteration += 1

        print(f"\n{'─'*40}")
        print(f"🔄 ITERATION {iteration}")
        print(f"{'─'*40}")

        response = client.chat.completions.create(
            model       = MODEL,
            messages    = messages,
            tools       = TOOLS,
            tool_choice = "auto",
            temperature = 0
        )

        message       = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        print(f"🧠 LLM DECISION: {finish_reason}")

        # --------------------------------------------------------
        # CASE 1: LLM wants to call a tool
        # --------------------------------------------------------
        if finish_reason == "tool_calls":

            messages.append(message)

            for tool_call in message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)

                print(f"🔧 TOOL REQUESTED: {tool_name}")

                reasoning_trace.append({
                    "iteration" : iteration,
                    "action"    : "tool_call",
                    "tool"      : tool_name,
                    "args"      : tool_args
                })

                tool_result = dispatch_tool(tool_name, tool_args)

                reasoning_trace.append({
                    "iteration"      : iteration,
                    "action"         : "observation",
                    "tool"           : tool_name,
                    "result_preview" : tool_result[:80]
                })

                messages.append({
                    "role"         : "tool",
                    "tool_call_id" : tool_call.id,
                    "content"      : tool_result
                })

        # --------------------------------------------------------
        # CASE 2: LLM is done — final answer
        # --------------------------------------------------------
        elif finish_reason == "stop":

            final_answer = message.content

            print(f"\n{'='*60}")
            print("✅ AGENT FINISHED")
            print(f"{'='*60}")
            print(final_answer)

            # ------------------------------------------------
            # Parse confidence from final answer
            # ------------------------------------------------
            confidence = parse_confidence(final_answer)
            print(f"\n📊 CONFIDENCE DETECTED: {confidence}")

            # ------------------------------------------------
            # Human override gate — fires only on LOW
            # ------------------------------------------------
            approved = human_override_gate(confidence, final_answer)

            # ------------------------------------------------
            # Print reasoning trace
            # ------------------------------------------------
            print(f"\n{'='*60}")
            print("📋 REASONING TRACE:")
            print(f"{'='*60}")
            for i, step in enumerate(reasoning_trace):
                print(f"\nStep {i+1}: {step['action'].upper()}")
                if step['action'] == 'tool_call':
                    print(f"  Tool : {step['tool']}")
                    print(f"  Args : {step['args']}")
                elif step['action'] == 'observation':
                    print(f"  Result : {step['result_preview']}")

            # ------------------------------------------------
            # Save to memory regardless of approval
            # ------------------------------------------------
            save_to_memory(user_goal, final_answer)

            return final_answer

        else:
            print(f"⚠️ Unexpected finish reason: {finish_reason}")
            break

    return "Agent reached maximum iterations without completing."

print("✅ Agent loop defined")
print("Run Cell 11 to fire the first investigation")

✅ Agent loop defined
Run Cell 11 to fire the first investigation


In [18]:
result_1 = run_agent(
    "Our purchase conversion has dropped significantly. "
    "Investigate the funnel, identify when the drop started, "
    "which user segments are affected, "
    "and what the most likely root cause is."
)


🤖 AGENT STARTED (Stage 2)
🎯 GOAL: Our purchase conversion has dropped significantly. Investigate the funnel, identify when the drop started, which user segments are affected, and what the most likely root cause is.

────────────────────────────────────────
🔄 ITERATION 1
────────────────────────────────────────
🧠 LLM DECISION: tool_calls
🔧 TOOL REQUESTED: get_funnel_summary

  ⚙️  TOOL CALLED  : get_funnel_summary
  📥 ARGUMENTS    : {}
  📤 RESULT PREVIEW: {"total_users": 10000, "signups": 7014, "activations": 3549, "purchases": 1079, "visit_to_signup_rate": 0.701, "signup_t...

────────────────────────────────────────
🔄 ITERATION 2
────────────────────────────────────────
🧠 LLM DECISION: tool_calls
🔧 TOOL REQUESTED: detect_anomalies

  ⚙️  TOOL CALLED  : detect_anomalies
  📥 ARGUMENTS    : {}
  📤 RESULT PREVIEW: {"average_activation_purchase_rate": 0.306, "anomaly_threshold": 0.214, "anomaly_days_count": 10, "anomaly_dates": ["202...

────────────────────────────────────────
🔄 ITERATIO

In [22]:
result_2 = run_agent(
    "We deployed a fix for the mobile checkout issue on January 28. "
    "Has the purchase conversion recovered after that date?"
)


🤖 AGENT STARTED (Stage 2)
🎯 GOAL: We deployed a fix for the mobile checkout issue on January 28. Has the purchase conversion recovered after that date?

────────────────────────────────────────
🔄 ITERATION 1
────────────────────────────────────────
🧠 LLM DECISION: tool_calls
🔧 TOOL REQUESTED: get_device_breakdown

  ⚙️  TOOL CALLED  : get_device_breakdown
  📥 ARGUMENTS    : {'start_date': '2025-01-29'}
  📤 RESULT PREVIEW: {"mobile": {"activations": 179, "purchases": 0, "activation_purchase_rate": 0.0}, "desktop": {"activations": 73, "purcha...

────────────────────────────────────────
🔄 ITERATION 2
────────────────────────────────────────
🧠 LLM DECISION: stop

✅ AGENT FINISHED
After the fix was deployed on January 28, 2025, the purchase conversion for mobile users has not recovered. The data shows that:

- Mobile users had 179 activations but 0 purchases, resulting in an activation to purchase rate of 0.0%.
- In contrast, desktop users had 73 activations and 33 purchases, leading to a

In [23]:
# CELL 12b — Check what memory actually says
print("=== CURRENT MEMORY CONTEXT ===")
print(get_memory_context())

=== CURRENT MEMORY CONTEXT ===
The investigations revealed a significant drop in purchase conversion rates, particularly affecting mobile users, which began on January 21, 2025. During this period, mobile users experienced a 0.0% activation to purchase rate, while desktop users maintained a healthy rate of 39.8%. Despite deploying a fix for the mobile checkout issue on January 28, 2025, the problem persisted, with mobile users continuing to show no purchases post-fix, while desktop users and other traffic sources demonstrated some level of conversion. The root cause was identified as a failure in the mobile user experience, leading to critical impairment in mobile conversions.


In [24]:
for case in FAILURE_CASES:
    print(f"\n{'🔴 '*20}")
    print(f"▶ FAILURE CASE {case['id']}: {case['label']}")
    print(f"  Expected: {case['expected_behaviour']}")
    print(f"{'🔴 '*20}")
    run_agent(case["goal"])


🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 
▶ FAILURE CASE F1: Vague question — no funnel signal
  Expected: Agent should note that its tools only cover funnel conversion metrics. It cannot diagnose pricing or competitive issues. Should flag LOW confidence and recommend manual investigation.
🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 

🤖 AGENT STARTED (Stage 2)
🎯 GOAL: Our revenue dropped last week. It might be pricing. It might be a competitor. Investigate and tell me the root cause.

────────────────────────────────────────
🔄 ITERATION 1
────────────────────────────────────────
🧠 LLM DECISION: stop

✅ AGENT FINISHED
CONFIDENCE: LOW  
REASON: This question is outside my tool coverage.

📊 CONFIDENCE DETECTED: LOW

⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  
⚠️  HUMAN REVIEW REQUIRED
⚠️  Agent confidence is LOW — recommendation may be unreliable
⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  

Agent conclusion:
CONFIDENCE: LOW  
REASON: This question is outside 

In [25]:
HUMAN_OVERRIDE_GUIDE = """
WHEN HUMANS SHOULD OVERRIDE THE AGENTIC SYSTEM
================================================

ALWAYS OVERRIDE WHEN:
1. Confidence is LOW — agent does not have enough data or tool
   coverage to conclude
2. Anomaly detection finds NO clear date — could be gradual
   degradation or a data pipeline issue, not a real product bug
3. Recommended action would affect >10% of users or >$100K revenue
4. Diagnosis contradicts what engineering already knows
   (e.g. no deploys happened on the anomaly date)
5. Two back-to-back investigations give different root causes
   for the same symptom

CONSIDER OVERRIDING WHEN (MEDIUM confidence):
1. Only one segment breakdown supports the conclusion —
   the other shows mixed signals
2. Anomaly period is very short (1-2 days) — could be noise,
   not a real issue
3. Past memory shows a different root cause for similar symptoms

DO NOT OVERRIDE WHEN (HIGH confidence):
1. Anomaly confirmed, clear start date, one segment clearly
   broken, other segment healthy
2. Consistent findings across BOTH device AND traffic breakdowns
3. Past memory aligns with current findings

WHAT TO DO AFTER OVERRIDING:
1. Note the reason for override in your incident log
2. Run the specific tool manually to sanity-check the data
3. Check with engineering about recent deploys on the anomaly date
4. If the agent was wrong, debug in this order:
   - Read reasoning trace — did agent call right tools?
   - Run tools manually — did they return correct data?
   - Check data freshness — is the dataset stale?
   - Check tool descriptions — did agent skip a needed tool?
   - Check system prompt — did guardrails fire incorrectly?
"""

print(HUMAN_OVERRIDE_GUIDE)


WHEN HUMANS SHOULD OVERRIDE THE AGENTIC SYSTEM

ALWAYS OVERRIDE WHEN:
1. Confidence is LOW — agent does not have enough data or tool
   coverage to conclude
2. Anomaly detection finds NO clear date — could be gradual
   degradation or a data pipeline issue, not a real product bug
3. Recommended action would affect >10% of users or >$100K revenue
4. Diagnosis contradicts what engineering already knows
   (e.g. no deploys happened on the anomaly date)
5. Two back-to-back investigations give different root causes
   for the same symptom

CONSIDER OVERRIDING WHEN (MEDIUM confidence):
1. Only one segment breakdown supports the conclusion —
   the other shows mixed signals
2. Anomaly period is very short (1-2 days) — could be noise,
   not a real issue
3. Past memory shows a different root cause for similar symptoms

DO NOT OVERRIDE WHEN (HIGH confidence):
1. Anomaly confirmed, clear start date, one segment clearly
   broken, other segment healthy
2. Consistent findings across BOTH device A